# Movie recommendations

This notebook compares movie metadata with TF-IDF and cosine similarity.

In [1]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data_path = "../data/cleaned_data.csv"
model_path = "../models/movie_recommender.joblib"
# Load the cleaned movie data
movies = pd.read_csv(data_path)
print("Movies available:", len(movies))
print("Metadata used: genres, keywords, overview, tagline")

Movies available: 4803
Metadata used: genres, keywords, overview, tagline


TF-IDF gives more weight to terms that help distinguish a movie from the rest of the catalog. Cosine similarity compares the resulting text vectors.

## Prepare text and calculate TF-IDF

In [2]:
# Fill missing text before combining the metadata
text_columns = ["genres_text", "keywords_text", "overview", "tagline"]
for column in text_columns:
    movies[column] = movies[column].fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

movies["combined_text"] = movies[text_columns].agg(" ".join, axis=1)
# Convert the movie text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=50000)
tfidf_matrix = vectorizer.fit_transform(movies["combined_text"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Movies with no text features:", int((tfidf_matrix.getnnz(axis=1) == 0).sum()))

TF-IDF matrix shape: (4803, 12574)
Movies with no text features: 0


## Recommendation function

In [3]:
def normalize_title(title):
    return " ".join(str(title).casefold().split())

# Create a lookup from normalized title to row number
title_to_index = {}
for index, title in enumerate(movies["title"]):
    title_to_index.setdefault(normalize_title(title), index)

# Find movies with metadata similar to the selected title
def recommend_movies(movie_title, n=5):
    movie_key = normalize_title(movie_title)
    if movie_key not in title_to_index:
        return "Movie not found in the dataset."
    if n <= 0:
        return pd.DataFrame(columns=["title", "genres", "similarity"])

    movie_index = title_to_index[movie_key]
    scores = cosine_similarity(tfidf_matrix[movie_index], tfidf_matrix).ravel()
    ranked_indices = scores.argsort()[::-1]
    # Remove the selected movie from its own recommendations
    ranked_indices = [i for i in ranked_indices if i != movie_index][:n]
    results = movies.loc[ranked_indices, ["title", "genres_text"]].copy()
    results = results.rename(columns={"genres_text": "genres"})
    results["similarity"] = scores[ranked_indices]
    return results.reset_index(drop=True)

print(recommend_movies("  THE DARK KNIGHT  ", 5))
print(recommend_movies("A title that is not in this dataset", 3))

                                     title                             genres  \
0                    The Dark Knight Rises  action | crime | drama | thriller   
1                           Batman Returns                   action | fantasy   
2                           Batman Forever           action | crime | fantasy   
3                            Batman Begins             action | crime | drama   
4  Batman: The Dark Knight Returns, Part 2                 action | animation   

   similarity  
0    0.438387  
1    0.360533  
2    0.352592  
3    0.339576  
4    0.337962  
Movie not found in the dataset.


## Sample recommendations

In [4]:
# Check recommendations for a few different movies
sample_queries = ["The Dark Knight", "Toy Story", "The Hangover", "The Shawshank Redemption", "The Conjuring"]
sample_results = {}
for query in sample_queries:
    result = recommend_movies(query, 5)
    sample_results[query] = result
    print(f"\nQuery: {query}")
    display(result if isinstance(result, str) else result.round({"similarity": 4}))


Query: The Dark Knight


,title,genres,similarity
0,The Dark Knight Rises,action | crime | drama | thriller,0.4384
1,Batman Returns,action | fantasy,0.3605
2,Batman Forever,action | crime | fantasy,0.3526
3,Batman Begins,action | crime | drama,0.3396
4,"Batman: The Dark Knight Returns, Part 2",action | animation,0.3380



Query: Toy Story


,title,genres,similarity
0,Toy Story 3,animation | family | comedy,0.5053
1,Toy Story 2,animation | comedy | family,0.4662
2,Small Soldiers,comedy | adventure | fantasy | science fiction...,0.2753
3,The 40 Year Old Virgin,comedy | romance,0.2712
4,Factory Girl,drama,0.1871



Query: The Hangover


,title,genres,similarity
0,Last Vegas,comedy,0.2545
1,The Hangover Part II,comedy,0.2314
2,Very Bad Things,comedy | crime | thriller,0.2090
3,What Happens in Vegas,comedy | romance,0.1410
4,Doug's 1st Movie,animation | family | comedy,0.1257



Query: The Shawshank Redemption


,title,genres,similarity
0,Escape Plan,action | thriller,0.2872
1,Double Jeopardy,crime | drama | mystery | thriller,0.2577
2,Civil Brand,crime | drama | thriller,0.2544
3,Standard Operating Procedure,documentary,0.2476
4,Prison,crime | drama | horror | thriller,0.2471



Query: The Conjuring


,title,genres,similarity
0,The Conjuring 2,horror,0.3988
1,Grave Encounters,thriller | horror,0.2798
2,The Apparition,horror | thriller,0.2292
3,The Amityville Horror,horror | thriller | drama,0.1949
4,Alone in the Dark,action | fantasy | horror | thriller,0.1625


## Qualitative evaluation

In [5]:
def genre_set(value):
    return {genre.strip() for genre in str(value).split("|") if genre.strip()}

# Compare the top results with the query movie's genres
evaluation_rows = []
for query, result in sample_results.items():
    if isinstance(result, str):
        continue
    query_index = title_to_index[normalize_title(query)]
    query_genres = genre_set(movies.loc[query_index, "genres_text"])
    for _, row in result.head(3).iterrows():
        shared = sorted(query_genres & genre_set(row["genres"]))
        evaluation_rows.append({
            "query": query,
            "recommendation": row["title"],
            "shared genres": ", ".join(shared) if shared else "none",
            "similarity": round(float(row["similarity"]), 4),
        })

evaluation = pd.DataFrame(evaluation_rows)
display(evaluation)
for query, rows in evaluation.groupby("query", sort=False):
    overlaps = rows.loc[rows["shared genres"] != "none", "shared genres"]
    observed_genres = sorted({genre.strip() for value in overlaps for genre in value.split(",")})
    print(f"{query}: {len(overlaps)} of {len(rows)} top recommendations share a genre; shared labels: {', '.join(observed_genres) if observed_genres else 'none'}.")
print("Evaluation is qualitative: the CSV has movie-level aggregates but no user-movie relevance labels.")

,query,recommendation,shared genres,similarity
0,The Dark Knight,The Dark Knight Rises,"action, crime, drama, thriller",0.4384
1,The Dark Knight,Batman Returns,action,0.3605
2,The Dark Knight,Batman Forever,"action, crime",0.3526
3,Toy Story,Toy Story 3,"animation, comedy, family",0.5053
4,Toy Story,Toy Story 2,"animation, comedy, family",0.4662
5,Toy Story,Small Soldiers,comedy,0.2753
6,The Hangover,Last Vegas,comedy,0.2545
7,The Hangover,The Hangover Part II,comedy,0.2314
8,The Hangover,Very Bad Things,comedy,0.2090
9,The Shawshank Redemption,Escape Plan,none,0.2872


The Dark Knight: 3 of 3 top recommendations share a genre; shared labels: action, crime, drama, thriller.
Toy Story: 3 of 3 top recommendations share a genre; shared labels: animation, comedy, family.
The Hangover: 3 of 3 top recommendations share a genre; shared labels: comedy.
The Shawshank Redemption: 2 of 3 top recommendations share a genre; shared labels: crime, drama.
The Conjuring: 3 of 3 top recommendations share a genre; shared labels: horror, thriller.
Evaluation is qualitative: the CSV has movie-level aggregates but no user-movie relevance labels.


## Save and reload the recommender

In [6]:
# Save the pieces needed to make recommendations later
model = {
    "titles": movies["title"].tolist(),
    "genres": movies["genres_text"].fillna("").tolist(),
    "title_to_index": title_to_index,
    "vectorizer": vectorizer,
    "tfidf_matrix": tfidf_matrix,
}
joblib.dump(model, model_path)
print("Saved model:", model_path)

loaded_model = joblib.load(model_path)
print("Loaded model with", len(loaded_model["titles"]), "movies")

loaded_index = loaded_model["title_to_index"][normalize_title("Toy Story")]
loaded_scores = cosine_similarity(
    loaded_model["tfidf_matrix"][loaded_index], loaded_model["tfidf_matrix"]
).ravel()
loaded_order = loaded_scores.argsort()[::-1]
loaded_order = [i for i in loaded_order if i != loaded_index][:5]
loaded_check = pd.DataFrame({
    "title": [loaded_model["titles"][i] for i in loaded_order],
    "similarity": loaded_scores[loaded_order],
})
display(loaded_check.round({"similarity": 4}))

Saved model: ../models/movie_recommender.joblib
Loaded model with 4803 movies


,title,similarity
0,Toy Story 3,0.5053
1,Toy Story 2,0.4662
2,Small Soldiers,0.2753
3,The 40 Year Old Virgin,0.2712
4,Factory Girl,0.1871
